# YOLO26s 심볼 검출 — 추가 증강 없는 첫 시험

기존 증강 이미지와 실제 도면 타일로 학습합니다. 원본 회사 이미지와 라벨은 로컬에서 처리합니다. OpenAI 키는 사용하지 않습니다.

- 학습 680장, 38클래스
- 전체 클래스 캡처 평가 73장, 38클래스
- 별도 도면 영역 평가 21장, 4클래스
- 640×640 입력, 수평 정답박스의 Detect 모델

38클래스 평가는 동일 물리 부품의 다른 캡처가 포함될 수 있습니다. 일부 클래스는 검증 1장뿐이므로 새 부품·새 도면 성능으로 해석하지 마세요.

## 이번 프로젝트에서 실제 실행한 결과

YOLO26s, 640×640, 50 epochs, batch 8, RTX 5060 Laptop GPU로 실행했습니다.
기존 증강본은 사용했고 새로운 온라인 증강은 껐습니다. `train_yolo26_noaug.py`를 별도 프로세스로 실행한 결과입니다.

| 평가 | 이미지 | 클래스 | mAP50 | mAP50–95 |
|---|---:|---:|---:|---:|
| 전체 클래스 캡처 | 73 | 38 | 65.1% | 54.4% |
| 실제 도면 영역 | 21 | 4 | 27.5% | 17.8% |

confidence 0.25 / 같은 클래스의 박스 IoU 0.5 기준으로 도면 영역의 정답 36개 중 3개를 맞췄고, 33개를 놓쳤습니다.
전체 클래스 평가는 알려진 부품의 다른 캡처이며 독립적인 새 도면 성능을 의미하지 않습니다.

현재 실제 도면 인식 성능은 부족합니다. 흰 배경 capture_075의 7번 2개는 검출했지만 같은 부품 영역의 어두운 capture_097은 놓쳤습니다.
배경·크기·표현 차이가 함께 있으므로 배경 하나만 원인으로 확정할 수 없습니다.
11번과 17번은 학습 정답박스가 각각 2개뿐입니다.

결과를 다시 확인하려면 아래 셀을 실행하세요. 이미 완료된 실행이 있어 재학습 없이 저장된 결과를 읽습니다.
새 학습은 RUN_NAME을 다른 이름으로 바꾸어 실행하세요.

[클래스별 결과](training_yolo26_noaug_20261008/baseline_noaug/reports/all38/per_class.csv)

[결과 설명](training_yolo26_noaug_20261008/baseline_noaug/RESULT.md)

공식 모델/API 참고: [Ultralytics YOLO26](https://docs.ultralytics.com/models/yolo26), [학습 모드](https://docs.ultralytics.com/modes/train).


## 1. 실행 커널과 프로젝트 위치

VS Code에서 `.venv-ml/Scripts/python.exe`를 커널로 선택하세요. 필요한 라이브러리와 공식 YOLO26s 가중치는 이 프로젝트에 준비되어 있습니다.

In [ ]:
from pathlib import Path
import sys, json, subprocess

PROJECT = Path(r"C:/Users/Win11Pro/Documents/ChatGPT/기업프로젝트 cad")
WORK = PROJECT / "symbol_roi640"
DATA = WORK / "dataset_yolo26_all38_eval_20261008"
PYTHON = PROJECT / ".venv-ml/Scripts/python.exe"
assert PYTHON.exists() and DATA.exists()
print("노트북 커널:", sys.executable)
print("학습에 사용할 Python:", PYTHON)
print(json.loads((DATA / "summary.json").read_text(encoding="utf-8")))

## 2. GPU 확인

별도 Python 프로세스로 실행해서 노트북이 GPU 메모리를 잡아두지 않게 합니다.

In [ ]:
check = "import torch,ultralytics; print(torch.__version__,ultralytics.__version__); print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0))"
subprocess.run([str(PYTHON), "-c", check], check=True, cwd=PROJECT)

## 3. 학습 설정

추가 증강은 사용하지 않습니다. `train_yolo26_noaug.py`의 `NoAugDetectionTrainer`는 실제 입력 변환을 LetterBox와 Format만 사용하도록 바꾸고 검사합니다. 단순히 `augment=False`만 주면 기본 학습 증강이 꺼지는 것은 아닙니다.

이번 시험은 정해진 50 epochs를 실행하고 마지막 epoch를 평가합니다. 학습 도중 검증 점수로 횟수를 바꾸지 않습니다. BF16을 사용하는 RTX 5060 GPU 환경의 설정입니다. Windows에서 안정적으로 실행하도록 데이터 로더 workers는 0입니다.

In [ ]:
EPOCHS = 50
BATCH = 8
RUN_NAME = "baseline_noaug"
RUN = WORK / "training_yolo26_noaug_20261008" / RUN_NAME
SCRIPT = WORK / "train_yolo26_noaug.py"
print("모델: YOLO26s Detect / 입력: 640×640")
print("학습 횟수:", EPOCHS, "배치:", BATCH)
print("결과 폴더:", RUN)

## 4. 학습 실행

이미 완료된 `baseline_noaug`가 있으면 결과를 재사용합니다. 다시 학습하고 싶으면 위의 `RUN_NAME`을 새로운 이름으로 바꾸세요. 중단된 폴더는 자동으로 덮어쓰지 않습니다. 실행 중에는 `progress.json`과 `console.log`에서 진행 상황을 볼 수 있습니다.

In [ ]:
if (RUN / "result_summary.json").exists():
    print("이미 완료된 학습 결과가 있어 재학습하지 않습니다.")
else:
    subprocess.run(
        [str(PYTHON), "-u", str(SCRIPT), "--epochs", str(EPOCHS),
         "--batch", str(BATCH), "--name", RUN_NAME],
        check=True, cwd=PROJECT,
    )

## 5. 실제 증강 설정과 평가 결과 확인

AP는 정답률이 아닙니다. mAP50은 박스 IoU 0.5 기준, mAP50–95는 여러 IoU 기준에서 평균한 검출 지표입니다. 클래스별 Precision/Recall은 Ultralytics의 평가 지점이며, 고정 confidence 0.25에서의 TP/FP/FN은 별도로 저장합니다.

In [ ]:
from IPython.display import display, Markdown
import pandas as pd

result = json.loads((RUN / "result_summary.json").read_text(encoding="utf-8"))
audit = json.loads((RUN / "no_augmentation_audit.json").read_text(encoding="utf-8"))
print("실제 학습 입력 변환:", audit)
display(pd.DataFrame(result["evaluations"]).T)
display(Markdown((RUN / "RESULT.md").read_text(encoding="utf-8")))

## 6. 클래스별로 잘 찾는지 확인

전체 38개 클래스와 4개 클래스 영역 결과를 따로 확인합니다. 검증 이미지가 한 장뿐인 클래스의 AP가 높아도 성능이 확실하다는 뜻은 아닙니다.

In [ ]:
all38 = pd.read_csv(RUN / "reports/all38/per_class.csv")
region4 = pd.read_csv(RUN / "reports/region4/per_class.csv")
print("38클래스 별도 캡처 평가")
display(all38.sort_values("mAP50"))
print("4클래스 영역 평가")
display(region4.sort_values("mAP50"))

## 7. 정답과 예측 박스 비교

왼쪽은 정답, 오른쪽은 모델 예측입니다. confidence 0.25, 같은 클래스와 박스 IoU 0.5 이상으로 매칭해서 TP/FP/FN을 표시했습니다. 라벨 누락이나 박스 경계 문제도 결과에 영향을 줄 수 있으므로 예측을 직접 확인하세요.

In [ ]:
from IPython.display import Image as DisplayImage
rows = json.loads((RUN / "reports/region4/predictions_conf025.json").read_text(encoding="utf-8"))
for row in sorted(rows, key=lambda r: (r["fn"] + r["fp"]), reverse=True)[:6]:
    print(row["image"], "TP:", row["tp"], "FP:", row["fp"], "FN:", row["fn"])
    display(DisplayImage(filename=str(RUN / "reports/region4" / row["preview"]), width=1100))

## 8. 학습 가중치로 다른 640 이미지 검출하기

다른 이미지 경로를 넣고 직접 실행하세요. PDF 전체는 고해상도로 렌더링한 뒤 겹치는 640 타일로 검출하고 좌표를 복원해야 합니다. 이 노트북은 첫 시험 학습과 평가 단계입니다.

In [ ]:
# 예: 학습 결과를 사용한 새 이미지 검출
# IMAGE_PATH = Path(r"C:/path/to/new_640_image.png")
# subprocess.run([
#     str(PYTHON), "-c",
#     "from ultralytics import YOLO; import sys; "
#     "YOLO(sys.argv[1]).predict(source=sys.argv[2], imgsz=640, device=0, "
#     "conf=0.25, save=True, project=sys.argv[3], name='new_image', exist_ok=True)",
#     str(RUN / "weights/last.pt"), str(IMAGE_PATH), str(RUN / "inference")
# ], check=True, cwd=PROJECT)